# Práctica 2. Hadoop MapReduce.

## SESION 2. MapReduce en Python con datasets reales. Uso de la biblioteca MRJob.

En esta práctica veremos usos de MapReduce sobre datasets reales y relativamente grandes. Desarrollaremos aplicaciones MapReduce en Python haciendo uso de la biblioteca MRJob.

---
## 1. Instalación de MRJob

Tal y como se ha explicado en teoría, la biblioteca *MRJob* permite escribir, de manera sencilla, programas MapReduce en Python. Una de las ventajas más interesantes es que un mismo programa MapReduce desarrollado con esta librería puede ejecutarse en modo:
1. Local. Esto es útil para pruebas rápidas o desarrollo.
2. Hadoop. Puede ser un clúster real o uno simulado.
3. En otras plataformas, como *Elastic MapReduce* (EMR) de AWS.

Esto hace que MrJob sea idonéo para **desarrollo híbrido** de aplicaciones MapReduce: pruebas locales rápidas para depurar el código y ejecución a escala en el clúster.

### 1.1 Instalación y ejecución en un ordenador con Python 

Para instalar MrJob en un ordenador vamos a usar uv para crear un nuevo .evn al igual que hicimos en las prácticas anteriores. En la carpeta environment/p2-s2-MapReduce se encuentran los ficheros necesarios para crear el nuevo .env. Además, en la capeta practicas/p2/2_2_mapreduce están los ficheros .py y los trozos del dataset que emplearemos.  
Se explica a continuación el proceso de generación del nuevo .env y así como la prueba de un ejemplo de MapReduce empleando MrJob en local. 

1. Hacemos pull del repo para traernos los ficheros que hemos añadido. Desde la raiz del repo hacemos:
```bash
git pull origin main
```

2. Podemos ver que nos ha bajado una nueva carpeta en environment/p2-s2-MapReduce. Creamos el nuevo .env en environment/p2-s2-MapReduce con:
```bash
uv lock
uv sync --locked
```
3. Activamos el entorno vitual en la consola desde la carpeta environment/p2-s2-MapReduce con:
```bash
source .venv/bin/activate
```

4. El usuario ya puede ejecutar los scripts con MrJob. Nótese que el *prompt* cambia a algo como:

```bash
(p3) ppd@ppd-Usuario:~$
```

    indicándonos que estamos usando el entorno virtual de ejecución `.venv`. Un entorno virtual permite instalar un conjunto de paquetes propio para cada usuario.

Una vez realizados los pasos anteriores, ya puedes ejecutar los ejemplos de MRJob:

```bash
python3 AplicacionMapReduce.py FICHERO_ENTRADA
```

5. Podemos probar con el ejemplo `word_count_mrjob.py` y el libro `pg16625.txt` que hay en la carpeta practicas/p2/2_2_mapreduce del repositorio. Se trata del ejemplo de MrJob que hay en las transparencia de teoría para contar palabras. Tenemos que usar el siguiente comando:
```bash
python3 word_count_mrjob.py pg16625.txt  -o word_count_mrjob_output
```
Se añade `-o` para guardar la salida en ficheros. En la salida podemos ver varias carpetas, tantas como reducer nos haya creado MrJob, con el siguiente formato: "palabra" \t contador.


### 1.2 Instalación y ejecución dentro del clúster Hadoop

Dentro del clúster Hadoop que desarrollamos en la sesión práctica anterior se tiene MrJob instalado y no es necesario hacer instalar ningún paquete adicional.
Para ejecutar el ejemplo `word_count_mrjob.py` dentro de Hadoop tenemos que llevar a cabo los siguientes pasos:

1. Copiar los dos ficheros, `word_count_mrjob.py` y `word_count_mrjob.py` al docker workbench con los siguientes comandos:
```bash
docker cp pg16625.txt workbench:/home/luser
docker cp pg16625.txt workbench:/home/luser
```

2. Conectarnos al workbench:
```bash
docker compose exec workbench bash
```

3. Cambiarnos al usuario luser
```bash
su – luser
```

4. Ya podemos ejecutar `word_count_mrjob.py` en Hadoop con el comando:
```bash
python3 word_count_mrjob.py -r hadoop pg16625.txt -o word_count_mrjob_output
```

5. Podemos ver la salida con 
```bash
hdfs dfs -ls word_count_mrjob_output
hdfs dfs -cat wc-mrjob-output/part-00000
```

---
## 2. Dataset reales

Antes de proponer diferentes ejercicios MapReduce para que pongas en práctica el uso de la librería *MrJob*, vamos a presentar el dataset de ficheros de entrada que vamos a utilizar, los cuales hemos descargado desde https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store

Como podéis ver, se trata de un log sobre el comportamiento de los usuarios cuando visitan la web de una tienda online. Para cada sesión de usuario, guardan en un log la sesión de usuario, el id de usuario, los productos que visualizan, si los añaden al carro de la compra, y por último, si los han comprado.  
Son 2 ficheros con varios GB de información separada en filas y cada fila separada por comas.
Para no usar esos dos fichero tan grandes, hemos sacado dos subconjuntos de estos datasets que se encuentran en la carpeta practicas/p2/2_2_mapreduce:  
- `2019-Oct_10k.txt`: contiene las primeras 10.000 filas del fichero 2019-Oct.csv y pesa 1.31 MB.
- `2019-Oct_600k.txt`: contiene las primeras 600.000 filas del fichero 2019-Oct.csv y pesa 78 MB.

 El orden y contenido de las columnas de estos dos ficheros es el siguiente:

- `event_time`: timestamp con la fecha en la que se produjo el evento.  
- `event_type`: tipo de evento. Puede ser view, cart, purchase.
- `product_id`: identificador del producto que está siendo visualizado.
- `category_id`: identificador de la categoría del producto.
- `category_code`: nombre descriptivo de la categoría a la que pertenece el producto.
- `brand`: marca del producto.
- `price`: precio. 
- `user_id`: identificador del usuario.
- `user_session`: sesión de usuario. Suele cambiar cada vez que se visualiza la web (para eso se usan las cookies)


A continuación se muestran las primeras líneas del fichero:
event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session  
2019-10-01 00:00:00 UTC,view,3900821,2053013552326770905,appliances.environment.water_heater,aqua,33.20,554748717,9333dfbd-b87a-4708-9857-6336556b0fcc  
2019-10-01 00:00:01 UTC,view,17200506,2053013559792632471,furniture.living_room.sofa,,543.10,519107250,566511c2-e2e3-422b-b695-cf8e6e792ca8  
2019-10-01 00:00:01 UTC,view,1307067,2053013558920217191,computers.notebook,lenovo,251.74,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713  

Os hemos dejado una versión reducida con XX mil líneas del fichero que ocupa XX MB para poder hacer las pruebas.

---
## 3. Ejercicios propuestos

En este apartado no sólo deberás demostrar todo lo aprendido hasta este momento sobre Hadoop MapReduce sino que también emplearás algunas características MapReduce no vistas hasta ahora. Por ello, encontrarás que los enunciados de los ejercicios mencionan conceptos nuevos. Es algo intencionado, pero no te asustes, te ofrecemos recursos suficientes para ser capaz de *aprender de manera autónoma*.  

**ENTREGA OBLIGATORIA**

A continuación encontrarás XX ejercicios que deberás resolver haciendo uso de la librería *MrJob*. La resolución de estos ejercicios deberás realizarla de **manera local y haciendo uso de la versión reducida de los datasets** anteriormente descritos (`2019-Oct_10k.txt` y `2019-Oct_600k.txt`). 

La resolución de estos ejercicios deberá ser incluida como **parte de la Tarea 2** de la asignatura, siguiendo las indicaciones que se recogen en el enunciado de la misma. Para conocer los criterios de evaluación y normas de entrega, **consulta el enunciado** de la misma.

**NOTA IMPORTANTE**

La memoria que debes entregar como resolución de la tarea 2 debe incluir un apartado donde muestres la resolución de cada ejercicio. La resolución no puede consistir en una mera secuencia de imágenes o líneas de código de los programas. Es necesario razonar el programa desarrollado, demostrar su correcto funcionamiento y proporcionar todos los comandos necesarios para probarlo. 

Además de la memoria, debes adjuntar un fichero comprimido que incluya los ficheros `.py` modificados con las soluciones de cada apartado.

*Actividad opcional 1*: es interesante que uses la versión que ocupa 78 MB para probar en el cluste de Hadoop.


### EJERCICIO 1: **Contador de eventos**. Resuelto. ej1_event_count.py

Desarrolla una aplicación MapReduce con MrJob que haga un conteo de eventos por tipo de evento. Como se puede observar, en la columna [1], event_type, hay eventos del tipo view, cart, purchase. En este primer ejemplo vamos aplicar MapReduce para ver cuántos eventos hay de cada tipo. Para ello tenemos que hacer Split de cada línea y yield del tipo de evento,1 en el mapper. El reducer simplemente hacemos  yield de key, sum(values)
Puedes ver y probar el código ej1_event_count.py donde viene este ejercicio completo y usarlo como ejemplo.  
La salida que debes de observar es similar a la que se muestra a continuación:  
"cart"	97  
"purchase"	118  
"view"	9784  


### EJERCICIO 2: **Contador de  usuarios**. ej2_user_count.py
Ahora, desarrolla una aplicación MapReduce con MrJob que haga un conteo de las interacciones para cada usuario. Lo haremos independientemente del tipo de evento. Es decir, para cada usuario tenemos que mostrar el número de interaciones con la plataforma. El código es similar al anterior pero empleando user_id en vez de event_type. La salida debe de ser similar a esta:
"306441847"	1  
"400972610"	5  
"441518854"	2  
"442188017"	1  
"445306726"	1  


### EJERCICIO 3: **Contador de usuarios por tipo de evento**. ej3_user_event_count.py
El ejercicio anterior hemos obtenido un listado de todos los usuarios, independientemente del tipo de evento. Pero para este ejercicio, hemos recibido una solicitud de mostrar tanto los usuario como el tipo de evento que han realizado. Esto lo hacemos para saber, por ejemplo, cuales han sido los usuarios que más productos han visualizado, pero también los quemás han comprado. Para ello, tenemos que hacer de nuevo un programa MaPreduce empleando MrJob, pero ahora el mapper tiene que emitir tanto el user_id, como el event_type. Es decir, algo del estilo (user_id, event_type), 1. La salida esperada es la siguiente:  
["306441847", "view"]	1  
["400972610", "view"]	5  
["441518854", "view"]	2  
["442188017", "view"]	1  
["445306726", "view"]	1  

### EJERCICIO 4: **Top 10 usuarios con más view**. ej4_top_users.py
Ahora, como hemos facilitado una lista con muchísimos usuarios, nos piden extraer los 10 usuarios con más interacciones, independientemente del tipo que sea. Para ello, tenemos que hacer dos reducers. Hay un ejemplo en las transparecencias de teoría donde se aplica este sistema. La salida esperada es la siguiente:
"528769528"	43  
"531063605"	31  
"538015502"	28  
"512975726"	27  
"543103721"	25  
"554754734"	24  
"514338857"	24  
"513840435"	24  
"512742880"	24  
"514190843"	23  

### EJERCICIO 5: **Top 10 productos con más view y descripción de los productos**. ej5_product_view_description.py. 
A continuación, han solicitado que mostremos los 10 productos más visualizados junto con el número de visualizaciones. Pero la solicitud incluye que mostremos también la categoría y la marca , es decir, los campos category_id y brand. En este caso, es similar al funcionamiento del ejercicio anterior en cuanto al uso de dos reducers. Sin embargo, requiere de modificar el mapper para hacer el yield de estos dos campos junto con el id de producto. OJO, solo queremos visualizaciones, por lo que se debe filtrar por eventos del tipo view. La salida sería algo de este estilo:  
"1004856"	[116, "electronics.smartphone", "samsung"]  
"1004767"	[115, "electronics.smartphone", "samsung"]  
"1005115"	[111, "electronics.smartphone", "apple"]  
"1005105"	[65, "electronics.smartphone", "apple"]  
"1004870"	[61, "electronics.smartphone", "samsung"]  
"1002544"	[60, "electronics.smartphone", "apple"]  
"1004833"	[57, "electronics.smartphone", "samsung"]  
"5100816"	[56, "", "xiaomi"]  
"1004249"	[52, "electronics.smartphone", "apple"]  
"1004750"	[41, "electronics.smartphone", "samsung"]  


